## text to speech with eleven lab ai

In [2]:
# import gradio as gr
import openai
import os
from datetime import datetime
import json
from pathlib import Path
import tempfile
import requests
import html

# PDF generation
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER
from dotenv import load_dotenv


# ─────────────────────────────────────────────
# Discovery Bible Study API helper
# ─────────────────────────────────────────────

DISCOVERY_API_URL = "https://discoverybiblestudy.org/daily/api/"

def fetch_verse_of_the_day() -> dict:
    """
    Fetches the real verse of the day from discoverybiblestudy.org.
    Returns a dict with keys: text, ref, date, url, verseUrl.
    Falls back to None on any error so the AI agent can still generate content.
    """
    try:
        response = requests.get(DISCOVERY_API_URL, timeout=10)
        response.raise_for_status()
        data = response.json()
        # The API returns HTML entities in the text – decode them
        data["text"] = html.unescape(data.get("text", ""))
        return data
    except Exception as e:
        print(f"[DiscoveryBibleStudy API] Could not fetch verse: {e}")
        return None


class BibleLanguageLearningSystem:
    """
    Agentic AI system for language learning through Bible study.
    Uses multiple specialised agents to create comprehensive lessons.
    """

    def __init__(
        self,
        api_key: str,
        target_language: str = "Spanish",
        model: str = "gpt-4o-mini",
    ):
        self.client = openai.OpenAI(api_key=api_key)
        self.model = model
        self.target_language = target_language

    # ──────────────────────────────────────────
    # Internal helpers
    # ──────────────────────────────────────────

    def _call_gpt(self, system_prompt: str, user_message: str, temperature: float = 1.0) -> str:
        """Helper method to call OpenAI API."""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message},
                ],
                temperature=temperature,
                max_tokens=4000,
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

    @staticmethod
    def _parse_json(response: str, fallback: dict) -> dict:
        """Strip markdown fences and parse JSON, returning fallback on failure."""
        try:
            if "```json" in response:
                json_str = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                json_str = response.split("```")[1].split("```")[0].strip()
            else:
                json_str = response.strip()
            return json.loads(json_str)
        except Exception:
            return fallback

    # ──────────────────────────────────────────
    # Agent 1 – Verse Retriever (now uses real API)
    # ──────────────────────────────────────────

    def agent_verse_retriever(self, language_level: str) -> dict:
        """
        Agent 1: Fetches the real verse of the day from discoverybiblestudy.org,
        then uses GPT to translate it and generate a meditation paragraph.
        """
        lang = self.target_language
        lang_lower = lang.lower()

        # --- Step 1: get the real verse ---
        api_data = fetch_verse_of_the_day()

        if api_data:
            verse_ref = api_data.get("ref", "Unknown")
            verse_english = api_data.get("text", "").strip()
            verse_date = api_data.get("date", datetime.now().strftime("%d %b %Y"))
            verse_source_url = api_data.get("verseUrl", "")
        else:
            # Graceful fallback if the API is unreachable
            verse_ref = "John 3:16"
            verse_english = "For God so loved the world that he gave his one and only Son, that whoever believes in him shall not perish but have eternal life."
            verse_date = datetime.now().strftime("%d %b %Y")
            verse_source_url = ""

        # --- Step 2: translate + meditation via GPT ---
        system_prompt = f"""You are a Bible study coordinator and language teacher.
You will receive a Bible verse in English. Your tasks:
1. Translate the verse accurately into {lang}.
2. Write a short meditation paragraph (2-3 sentences) in English appropriate for {language_level} learners.
3. Write the same meditation paragraph in {lang} at {language_level} level (clear, educational language).
4. Keep total word count under 1000 words.

Return ONLY valid JSON with these exact keys:
- "verse_text_english": the English verse as provided
- "verse_text_{lang_lower}": the verse translated into {lang}
- "meditation_english": meditation in English
- "meditation_{lang_lower}": meditation in {lang}"""

        user_message = (
            f"Verse reference: {verse_ref}\n"
            f"English verse text: {verse_english}\n"
            f"Target language level: {language_level}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.7)

        fallback = {
            "verse_text_english": verse_english,
            f"verse_text_{lang_lower}": verse_english,
            "meditation_english": "Reflect on this verse and apply it to your daily life.",
            f"meditation_{lang_lower}": "Reflexiona sobre este verso y aplícalo a tu vida diaria.",
        }

        result = self._parse_json(response, fallback)

        # Always inject real API metadata
        result["verse_reference"] = verse_ref
        result["verse_date"] = verse_date
        result["verse_source_url"] = verse_source_url

        return result

    # ──────────────────────────────────────────
    # Agent 2 – Content Creator
    # ──────────────────────────────────────────

    def agent_content_creator(self, verse_data: dict, language_level: str) -> dict:
        """Agent 2: Creates reading comprehension paragraph."""
        lang_lower = self.target_language.lower()

        system_prompt = f"""You are a language learning content creator.
Create a reading comprehension paragraph (150-200 words) in {self.target_language}.

Requirements:
- Appropriate for {language_level} level
- Include theological insights and practical applications
- Use clear, educational language
- Natural pronunciation-friendly text (avoid complex punctuation)

Return ONLY valid JSON with:
- "reading_text_{lang_lower}": Reading text in {self.target_language}
- "reading_text_english": Reading text in English
- "key_vocabulary": Array of important vocabulary words"""

        user_message = (
            f"Verse: {verse_data.get('verse_reference', 'N/A')}\n"
            f"Text: {verse_data.get(f'verse_text_{lang_lower}', '')}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.8)
        return self._parse_json(
            response,
            {
                f"reading_text_{lang_lower}": response[:300],
                "reading_text_english": "Reading comprehension text",
                "key_vocabulary": ["faith", "love", "grace"],
            },
        )

    # ──────────────────────────────────────────
    # Agent 3 – Lesson Designer
    # ──────────────────────────────────────────

    def agent_lesson_designer(self, verse_data: dict, reading_data: dict, language_level: str) -> dict:
        """Agent 3: Designs comprehensive lesson exercises."""
        lang_lower = self.target_language.lower()

        system_prompt = f"""You are an expert language lesson designer for {self.target_language}.
Create a comprehensive lesson for {language_level} level including:

1. READING: 4-5 comprehension questions about the reading text (in {self.target_language})
2. WRITING: 3 writing prompts related to the theme (in {self.target_language})
3. LISTENING: 4 questions about what students should listen for in the audio (in {self.target_language})
4. SPEAKING: 3 speaking prompts for oral practice (in {self.target_language})
5. FILLING: 3-4 fill-in-the-blank sentences using vocabulary from the reading (in {self.target_language})
   - Use ___ to indicate where the word should go
   - Make blanks appropriate for {language_level} level

Return ONLY valid JSON with:
- "reading_exercises": Array of objects with "question" field
- "writing_exercises": Array of objects with "question" field
- "listening_exercises": Array of objects with "question" field
- "speaking_exercises": Array of objects with "question" field
- "filling_exercises": Array of objects with "question" field (sentences with ___ for blanks)

Return ONLY the JSON object, no additional text."""

        user_message = (
            f"Verse: {verse_data.get('verse_reference')}\n"
            f"Reading: {reading_data.get(f'reading_text_{lang_lower}', '')[:200]}\n"
            f"Vocabulary: {reading_data.get('key_vocabulary', [])}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.7)
        return self._parse_json(
            response,
            {
                "reading_exercises": [{"question": "¿Cuál es el tema principal del texto?"}],
                "writing_exercises": [{"question": "Escribe sobre tu experiencia personal con este tema."}],
                "listening_exercises": [{"question": "¿Qué palabras clave escuchaste?"}],
                "speaking_exercises": [{"question": "Explica el significado del verso en tus propias palabras."}],
                "filling_exercises": [{"question": "La ___ es importante en la vida cristiana."}],
            },
        )

    # ──────────────────────────────────────────
    # Agent 4 – Answer Key Generator
    # ──────────────────────────────────────────

    def agent_answer_key_generator(self, lesson_data: dict, verse_data: dict, reading_data: dict) -> dict:
        """Agent 4: Generates answer key."""
        lang_lower = self.target_language.lower()

        system_prompt = f"""You are an answer key generator for {self.target_language}.
Provide detailed answers and model responses in {self.target_language}.

For filling exercises, provide ONLY the word(s) that should fill the blank(s).

Return ONLY valid JSON with:
- "reading_exercises": Array with "answer" and "explanation"
- "writing_exercises": Array with "answer" (model response) and "explanation"
- "listening_exercises": Array with "answer" (key points to listen for) and "explanation"
- "speaking_exercises": Array with "answer" (sample response) and "explanation"
- "filling_exercises": Array with "answer" (the missing word/phrase ONLY) and "explanation"

Return ONLY the JSON object, no additional text."""

        user_message = (
            f"Exercises: {json.dumps(lesson_data, ensure_ascii=False)[:500]}\n"
            f"Reading context: {reading_data.get(f'reading_text_{lang_lower}', '')[:300]}\n"
            f"Vocabulary: {reading_data.get('key_vocabulary', [])}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.5)
        return self._parse_json(
            response,
            {
                "reading_exercises": [{"answer": "El tema principal es...", "explanation": "Se encuentra en el párrafo principal"}],
                "writing_exercises": [{"answer": "Ejemplo de respuesta modelo", "explanation": "Respuesta modelo"}],
                "listening_exercises": [{"answer": "Palabras clave: fe, amor, esperanza", "explanation": "Escuchar atentamente"}],
                "speaking_exercises": [{"answer": "El verso significa que...", "explanation": "Guía de conversación"}],
                "filling_exercises": [{"answer": "fe", "explanation": "La palabra correcta es 'fe' según el contexto"}],
            },
        )

    # ──────────────────────────────────────────
    # Agent 5 – Grammar Lesson (NEW)
    # ──────────────────────────────────────────

    def agent_grammar_lesson(self, reading_data: dict, language_level: str) -> dict:
        """
        Agent 5 (NEW): Analyses the reading comprehension text and builds a
        targeted grammar mini-lesson appropriate for the learner's level.
        """
        lang = self.target_language
        lang_lower = lang.lower()
        reading_text = reading_data.get(f"reading_text_{lang_lower}", "")

        system_prompt = f"""You are an expert {lang} grammar teacher for {language_level} learners.
Analyse the reading text provided and identify 1-2 grammar points that are:
- Present in the reading text
- Appropriate and useful for {language_level} learners
- Practical for everyday communication

Build a concise grammar mini-lesson that includes:
1. A clear explanation of each grammar point in English
2. The grammar rule stated simply
3. 3-4 example sentences taken from or inspired by the reading text (in {lang} with English translations)
4. 3 practice exercises for the learner to complete

Return ONLY valid JSON with:
- "grammar_points": Array of objects, each with:
    - "name": Short name of the grammar point (e.g. "Present Perfect", "Ser vs Estar")
    - "explanation": Clear explanation in English (2-4 sentences)
    - "rule": The rule in simple terms
    - "examples": Array of objects with "sentence_{lang_lower}" and "sentence_english"
- "grammar_exercises": Array of objects with "instruction" and "exercise" fields
- "grammar_tips": A helpful tip string for {language_level} learners in English

Return ONLY the JSON object, no additional text."""

        user_message = (
            f"Reading text in {lang}:\n{reading_text}\n\n"
            f"Learner level: {language_level}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.6)
        return self._parse_json(
            response,
            {
                "grammar_points": [
                    {
                        "name": "Basic Grammar",
                        "explanation": "Grammar explanation based on the reading.",
                        "rule": "See examples below.",
                        "examples": [
                            {f"sentence_{lang_lower}": "Ejemplo.", "sentence_english": "Example."}
                        ],
                    }
                ],
                "grammar_exercises": [
                    {"instruction": "Rewrite the sentence using the grammar point.", "exercise": "Practice sentence here."}
                ],
                "grammar_tips": f"Focus on understanding the grammar pattern in context at {language_level} level.",
            },
        )

    # ──────────────────────────────────────────
    # Agent 6 – Role Play Creator (NEW)
    # ──────────────────────────────────────────

    def agent_roleplay_creator(self, verse_data: dict, reading_data: dict, language_level: str) -> dict:
        """
        Agent 6 (NEW): Creates a role-play scenario based on the day's theme so
        learners can practise common daily conversation at their level.
        """
        lang = self.target_language
        lang_lower = lang.lower()
        theme = verse_data.get("verse_reference", "today's reading")

        system_prompt = f"""You are a creative {lang} conversation coach for {language_level} learners.
Design an engaging role-play scenario inspired by the spiritual theme of the lesson.
The scenario must involve a realistic, everyday situation (e.g. at a café, at work, with a neighbour)
that a {language_level} learner could naturally encounter.

Include:
1. A clear scenario description (in English)
2. Two character roles the learner can choose from
3. A sample dialogue (8-12 exchanges) in {lang} with English translations side by side
4. 5-8 useful phrases / expressions from the dialogue that the learner should memorise
5. 3 follow-up conversation challenges (variations the learner can try)

Return ONLY valid JSON with:
- "scenario_title": Short catchy title
- "scenario_description": Description in English (3-5 sentences)
- "characters": Array of 2 objects with "name" and "role" fields
- "dialogue": Array of objects with "speaker", "line_{lang_lower}", "line_english"
- "useful_phrases": Array of objects with "phrase_{lang_lower}", "phrase_english", "notes"
- "conversation_challenges": Array of objects with "challenge" (in English)

Return ONLY the JSON object, no additional text."""

        user_message = (
            f"Lesson theme / verse: {theme}\n"
            f"Reading topic: {reading_data.get('reading_text_english', '')[:200]}\n"
            f"Learner level: {language_level}\n"
            f"Target language: {lang}"
        )

        response = self._call_gpt(system_prompt, user_message, temperature=0.85)
        return self._parse_json(
            response,
            {
                "scenario_title": "Daily Conversation Practice",
                "scenario_description": "Practice everyday conversation inspired by today's lesson.",
                "characters": [
                    {"name": "Person A", "role": "Learner"},
                    {"name": "Person B", "role": "Native Speaker"},
                ],
                "dialogue": [
                    {
                        "speaker": "Person A",
                        f"line_{lang_lower}": "Hola, ¿cómo estás?",
                        "line_english": "Hello, how are you?",
                    }
                ],
                "useful_phrases": [
                    {
                        f"phrase_{lang_lower}": "¿Cómo estás?",
                        "phrase_english": "How are you?",
                        "notes": "Common greeting",
                    }
                ],
                "conversation_challenges": [
                    {"challenge": "Try the dialogue without looking at the translations."}
                ],
            },
        )

    # ──────────────────────────────────────────
    # Agent 7 – TTS Generator
    # ──────────────────────────────────────────

    def agent_tts_generator(self, reading_text: str, language_level: str) -> str:
        """Agent 7: Generates text-to-speech audio using OpenAI."""
        try:
            temp_dir = "lessons"
            os.makedirs(temp_dir, exist_ok=True)

            audio_filename = (
                f"reading_audio_{self.target_language}_{language_level}_"
                f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.mp3"
            )
            audio_path = os.path.join(temp_dir, audio_filename)
            print(f"Generating OpenAI TTS audio: {audio_path}")

            response = self.client.audio.speech.create(
                model="gpt-4o-mini-tts",
                voice="alloy",
                input=reading_text,
                response_format="mp3",
            )

            with open(audio_path, "wb") as f:
                f.write(response.read())

            return audio_path

        except Exception as e:
            print(f"TTS Generation Error: {str(e)}")
            return None

    # ──────────────────────────────────────────
    # PDF generation
    # ──────────────────────────────────────────

    def generate_pdf(self, lesson_content: dict, filename: str = None):
        """Generate PDF with all lesson content."""
        filename = (
            f"bible_lesson_{self.target_language}_{lesson_content['level']}"
            f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
        )
        temp_dir = "lessons"
        os.makedirs(temp_dir, exist_ok=True)
        filepath = os.path.join(temp_dir, filename)
        print(f"PDF filepath: {filepath}")

        doc = SimpleDocTemplate(
            filepath, pagesize=letter,
            rightMargin=72, leftMargin=72,
            topMargin=72, bottomMargin=18,
        )

        elements = []
        styles = getSampleStyleSheet()

        title_style = ParagraphStyle(
            "CustomTitle",
            parent=styles["Heading1"],
            fontSize=24,
            textColor="darkblue",
            spaceAfter=30,
            alignment=TA_CENTER,
        )
        lang_lower = self.target_language.lower()

        # ── Title ──
        elements.append(Paragraph(f"Bible Language Learning Lesson<br/>{self.target_language}", title_style))
        elements.append(Spacer(1, 0.2 * inch))

        # ── Date / Level ──
        verse_data = lesson_content.get("verse_data", {})
        verse_date = verse_data.get("verse_date", datetime.now().strftime("%B %d, %Y"))
        date_text = f"Date: {verse_date}<br/>Level: {lesson_content.get('level', 'B1')}"
        elements.append(Paragraph(date_text, styles["BodyText"]))
        elements.append(Spacer(1, 0.3 * inch))

        # ── Verse ──
        elements.append(Paragraph("📖 Verse of the Day", styles["Heading2"]))
        elements.append(Spacer(1, 0.1 * inch))

        verse_ref = verse_data.get("verse_reference", "N/A")
        elements.append(Paragraph(f"<b>{verse_ref}</b>", styles["BodyText"]))

        verse_text = verse_data.get(f"verse_text_{lang_lower}", "N/A")
        elements.append(Paragraph(f"<i>{verse_text}</i>", styles["BodyText"]))
        elements.append(Spacer(1, 0.2 * inch))

        # Source URL
        source_url = verse_data.get("verse_source_url", "")
        if source_url:
            elements.append(Paragraph(f"Source: {source_url}", styles["BodyText"]))
            elements.append(Spacer(1, 0.1 * inch))

        # Meditation
        meditation = verse_data.get(f"meditation_{lang_lower}", "")
        if meditation:
            elements.append(Paragraph(f"<b>Meditation:</b> {meditation}", styles["BodyText"]))
            elements.append(Spacer(1, 0.3 * inch))

        # ── Reading ──
        reading_data = lesson_content.get("reading_data", {})
        elements.append(Paragraph("📚 Reading Comprehension", styles["Heading2"]))
        elements.append(Spacer(1, 0.1 * inch))
        reading_text = reading_data.get(f"reading_text_{lang_lower}", "N/A")
        elements.append(Paragraph(reading_text, styles["BodyText"]))
        elements.append(Spacer(1, 0.3 * inch))

        if lesson_content.get("audio_path"):
            elements.append(Paragraph("<b>🔊 Audio available for listening exercise</b>", styles["BodyText"]))
            elements.append(Spacer(1, 0.2 * inch))

        vocab = reading_data.get("key_vocabulary", [])
        if vocab:
            elements.append(Paragraph("<b>Key Vocabulary:</b>", styles["BodyText"]))
            vocab_text = ", ".join(vocab) if isinstance(vocab, list) else str(vocab)
            elements.append(Paragraph(vocab_text, styles["BodyText"]))
            elements.append(Spacer(1, 0.3 * inch))

        elements.append(PageBreak())

        # ── Standard Exercises ──
        lesson_data = lesson_content.get("lesson_data", {})
        self._add_exercises(elements, "📖 Reading Exercises", lesson_data.get("reading_exercises", []), styles)
        self._add_exercises(elements, "✍️ Writing Exercises", lesson_data.get("writing_exercises", []), styles)
        self._add_exercises(elements, "👂 Listening Exercises", lesson_data.get("listening_exercises", []), styles)
        self._add_exercises(elements, "🗣️ Speaking Exercises", lesson_data.get("speaking_exercises", []), styles)
        self._add_exercises(elements, "✏️ Fill-in-the-Blank Exercises", lesson_data.get("filling_exercises", []), styles)

        elements.append(PageBreak())

        # ── Grammar Lesson ──
        grammar_data = lesson_content.get("grammar_data", {})
        elements.append(Paragraph("📝 Grammar Lesson", styles["Heading2"]))
        elements.append(Spacer(1, 0.1 * inch))

        for gp in grammar_data.get("grammar_points", []):
            elements.append(Paragraph(f"<b>{gp.get('name', '')}</b>", styles["Heading3"]))
            elements.append(Paragraph(gp.get("explanation", ""), styles["BodyText"]))
            elements.append(Paragraph(f"<i>Rule: {gp.get('rule', '')}</i>", styles["BodyText"]))
            elements.append(Spacer(1, 0.1 * inch))
            for ex in gp.get("examples", []):
                tl = ex.get(f"sentence_{lang_lower}", "")
                en = ex.get("sentence_english", "")
                elements.append(Paragraph(f"• {tl} <i>({en})</i>", styles["BodyText"]))
            elements.append(Spacer(1, 0.2 * inch))

        tips = grammar_data.get("grammar_tips", "")
        if tips:
            elements.append(Paragraph(f"<b>💡 Tip:</b> {tips}", styles["BodyText"]))
            elements.append(Spacer(1, 0.2 * inch))

        elements.append(Paragraph("<b>Grammar Exercises</b>", styles["Heading3"]))
        for i, ex in enumerate(grammar_data.get("grammar_exercises", []), 1):
            elements.append(Paragraph(f"{i}. {ex.get('instruction', '')} — {ex.get('exercise', '')}", styles["BodyText"]))
            elements.append(Spacer(1, 0.15 * inch))

        elements.append(PageBreak())

        # ── Role Play ──
        rp = lesson_content.get("roleplay_data", {})
        elements.append(Paragraph("🎭 Role Play", styles["Heading2"]))
        elements.append(Spacer(1, 0.1 * inch))
        elements.append(Paragraph(f"<b>{rp.get('scenario_title', '')}</b>", styles["Heading3"]))
        elements.append(Paragraph(rp.get("scenario_description", ""), styles["BodyText"]))
        elements.append(Spacer(1, 0.2 * inch))

        characters = rp.get("characters", [])
        if characters:
            char_text = " | ".join([f"<b>{c['name']}</b>: {c['role']}" for c in characters])
            elements.append(Paragraph(char_text, styles["BodyText"]))
            elements.append(Spacer(1, 0.15 * inch))

        elements.append(Paragraph("<b>Sample Dialogue</b>", styles["Heading3"]))
        for line in rp.get("dialogue", []):
            speaker = line.get("speaker", "")
            tl_line = line.get(f"line_{lang_lower}", "")
            en_line = line.get("line_english", "")
            elements.append(Paragraph(f"<b>{speaker}:</b> {tl_line} <i>({en_line})</i>", styles["BodyText"]))
            elements.append(Spacer(1, 0.1 * inch))

        elements.append(Spacer(1, 0.2 * inch))
        elements.append(Paragraph("<b>Useful Phrases</b>", styles["Heading3"]))
        for phrase in rp.get("useful_phrases", []):
            tl_p = phrase.get(f"phrase_{lang_lower}", "")
            en_p = phrase.get("phrase_english", "")
            note = phrase.get("notes", "")
            elements.append(Paragraph(f"• {tl_p} — <i>{en_p}</i>{f' ({note})' if note else ''}", styles["BodyText"]))
            elements.append(Spacer(1, 0.1 * inch))

        elements.append(Spacer(1, 0.2 * inch))
        elements.append(Paragraph("<b>Conversation Challenges</b>", styles["Heading3"]))
        for i, ch in enumerate(rp.get("conversation_challenges", []), 1):
            elements.append(Paragraph(f"{i}. {ch.get('challenge', '')}", styles["BodyText"]))
            elements.append(Spacer(1, 0.12 * inch))

        elements.append(PageBreak())

        # ── Answer Key ──
        elements.append(Paragraph("✅ Answer Key", styles["Heading2"]))
        elements.append(Spacer(1, 0.2 * inch))

        answers = lesson_content.get("answers", {})
        self._add_answers(elements, "Reading Answers", answers.get("reading_exercises", []), styles)
        self._add_answers(elements, "Writing Answers", answers.get("writing_exercises", []), styles)
        self._add_answers(elements, "Listening Answers", answers.get("listening_exercises", []), styles)
        self._add_answers(elements, "Speaking Answers", answers.get("speaking_exercises", []), styles)
        self._add_answers(elements, "Fill-in-the-Blank Answers", answers.get("filling_exercises", []), styles)

        doc.build(elements)
        return filepath

    def _add_exercises(self, elements, title, exercises, styles):
        elements.append(Paragraph(title, styles["Heading2"]))
        elements.append(Spacer(1, 0.1 * inch))
        if isinstance(exercises, list):
            for i, ex in enumerate(exercises, 1):
                question = ex.get("question", str(ex)) if isinstance(ex, dict) else str(ex)
                elements.append(Paragraph(f"{i}. {question}", styles["BodyText"]))
                elements.append(Spacer(1, 0.15 * inch))
        elements.append(Spacer(1, 0.3 * inch))

    def _add_answers(self, elements, title, answers, styles):
        elements.append(Paragraph(f"<b>{title}</b>", styles["Heading3"]))
        elements.append(Spacer(1, 0.1 * inch))
        if isinstance(answers, list):
            for i, ans in enumerate(answers, 1):
                if isinstance(ans, dict):
                    answer = ans.get("answer", "")
                    explanation = ans.get("explanation", "")
                    text = f"{i}. <b>{answer}</b>"
                    if explanation:
                        text += f" <i>({explanation})</i>"
                else:
                    text = f"{i}. {str(ans)}"
                elements.append(Paragraph(text, styles["BodyText"]))
                elements.append(Spacer(1, 0.1 * inch))
        elements.append(Spacer(1, 0.2 * inch))

    # ──────────────────────────────────────────
    # Orchestrator
    # ──────────────────────────────────────────

    def run_full_lesson_generation(self, language_level: str = "B1"):
        """Generate complete lesson with progress updates."""
        print("0.00  Starting lesson generation...")

        # Step 1 – Verse (real API + translation)
        print("0.10  📖 Fetching verse of the day from discoverybiblestudy.org ...")
        verse_data = self.agent_verse_retriever(language_level)

        # Step 2 – Reading comprehension
        print("0.25  📚 Creating reading comprehension...")
        reading_data = self.agent_content_creator(verse_data, language_level)

        # Step 3 – Lesson exercises
        print("0.38  🎓 Designing lesson exercises...")
        lesson_data = self.agent_lesson_designer(verse_data, reading_data, language_level)

        # Step 4 – Answer key
        print("0.50  ✅ Generating answer key...")
        answers = self.agent_answer_key_generator(lesson_data, verse_data, reading_data)

        # Step 5 – Grammar lesson (NEW)
        print("0.62  📝 Building grammar lesson...")
        grammar_data = self.agent_grammar_lesson(reading_data, language_level)

        # Step 6 – Role play (NEW)
        print("0.74  🎭 Creating role-play scenario...")
        roleplay_data = self.agent_roleplay_creator(verse_data, reading_data, language_level)

        # Step 7 – Audio
        print("0.84  🔊 Generating audio...")
        audio_path = None
        reading_text = reading_data.get(f"reading_text_{self.target_language.lower()}", "")
        if reading_text:
            audio_path = self.agent_tts_generator(reading_text, language_level)

        # Step 8 – PDF
        print("0.93  📄 Creating PDF...")
        lesson_content = {
            "level": language_level,
            "verse_data": verse_data,
            "reading_data": reading_data,
            "lesson_data": lesson_data,
            "answers": answers,
            "grammar_data": grammar_data,
            "roleplay_data": roleplay_data,
            "audio_path": audio_path,
        }
        pdf_path = self.generate_pdf(lesson_content)

        print("1.00  ✨ Complete!")
        return lesson_content, pdf_path, audio_path


# ─────────────────────────────────────────────
# Display helpers
# ─────────────────────────────────────────────

def format_lesson_display(lesson_content):
    """Format lesson content for display."""
    verse_data = lesson_content.get("verse_data", {})
    reading_data = lesson_content.get("reading_data", {})
    lesson_data = lesson_content.get("lesson_data", {})
    grammar_data = lesson_content.get("grammar_data", {})
    roleplay_data = lesson_content.get("roleplay_data", {})

    lang_keys = [k for k in verse_data.keys() if k.startswith("verse_text_") and k != "verse_text_english"]
    verse_lang_key = lang_keys[0] if lang_keys else "verse_text_english"
    meditation_lang_key = verse_lang_key.replace("verse_text_", "meditation_")
    reading_lang_key = verse_lang_key.replace("verse_text_", "reading_text_")
    lang_lower = verse_lang_key.replace("verse_text_", "")

    source_url = verse_data.get("verse_source_url", "")
    source_line = f"\n[Read online]({source_url})" if source_url else ""

    output = f"""
# 📖 Verse of the Day — {verse_data.get('verse_date', '')}

**{verse_data.get('verse_reference', 'N/A')}**

*{verse_data.get(verse_lang_key, verse_data.get('verse_text_english', 'N/A'))}*{source_line}

**Meditation:**
{verse_data.get(meditation_lang_key, verse_data.get('meditation_english', 'N/A'))}

---

# 📚 Reading Comprehension

{reading_data.get(reading_lang_key, reading_data.get('reading_text_english', 'N/A'))}

**Key Vocabulary:** {', '.join(reading_data.get('key_vocabulary', []))}

---

# 📝 Grammar Lesson

"""
    for gp in grammar_data.get("grammar_points", []):
        output += f"## {gp.get('name', '')}\n"
        output += f"{gp.get('explanation', '')}\n\n"
        output += f"**Rule:** {gp.get('rule', '')}\n\n"
        output += "**Examples:**\n"
        for ex in gp.get("examples", []):
            tl = ex.get(f"sentence_{lang_lower}", "")
            en = ex.get("sentence_english", "")
            output += f"- {tl} *('{en}')*\n"
        output += "\n"

    tips = grammar_data.get("grammar_tips", "")
    if tips:
        output += f"💡 **Tip:** {tips}\n\n"

    output += "**Grammar Exercises:**\n"
    for i, ex in enumerate(grammar_data.get("grammar_exercises", []), 1):
        output += f"{i}. {ex.get('instruction', '')} — *{ex.get('exercise', '')}*\n"

    output += f"""

---

# 🎭 Role Play — {roleplay_data.get('scenario_title', '')}

{roleplay_data.get('scenario_description', '')}

**Characters:** {' | '.join([f"{c['name']} ({c['role']})" for c in roleplay_data.get('characters', [])])}

### 💬 Sample Dialogue
"""
    for line in roleplay_data.get("dialogue", []):
        speaker = line.get("speaker", "")
        tl_line = line.get(f"line_{lang_lower}", "")
        en_line = line.get("line_english", "")
        output += f"**{speaker}:** {tl_line} *('{en_line}')*\n\n"

    output += "\n### 🗝️ Useful Phrases\n"
    for phrase in roleplay_data.get("useful_phrases", []):
        tl_p = phrase.get(f"phrase_{lang_lower}", "")
        en_p = phrase.get("phrase_english", "")
        note = phrase.get("notes", "")
        output += f"- **{tl_p}** — {en_p}" + (f" *({note})*" if note else "") + "\n"

    output += "\n### 🔁 Conversation Challenges\n"
    for i, ch in enumerate(roleplay_data.get("conversation_challenges", []), 1):
        output += f"{i}. {ch.get('challenge', '')}\n"

    output += f"""

---

# 📝 Exercises

## 📖 Reading Exercises
"""
    for i, ex in enumerate(lesson_data.get("reading_exercises", []), 1):
        output += f"{i}. {ex.get('question', str(ex))}\n"

    output += "\n## ✍️ Writing Exercises\n"
    for i, ex in enumerate(lesson_data.get("writing_exercises", []), 1):
        output += f"{i}. {ex.get('question', str(ex))}\n"

    output += "\n## 👂 Listening Exercises\n"
    for i, ex in enumerate(lesson_data.get("listening_exercises", []), 1):
        output += f"{i}. {ex.get('question', str(ex))}\n"

    output += "\n## 🗣️ Speaking Exercises\n"
    for i, ex in enumerate(lesson_data.get("speaking_exercises", []), 1):
        output += f"{i}. {ex.get('question', str(ex))}\n"

    output += "\n## ✏️ Fill-in-the-Blank Exercises\n"
    for i, ex in enumerate(lesson_data.get("filling_exercises", []), 1):
        output += f"{i}. {ex.get('question', str(ex))}\n"

    return output


# ─────────────────────────────────────────────
# Gradio entry point
# ─────────────────────────────────────────────

def generate_lesson(api_key, language, level, model):
    """Main function called by Gradio interface."""
    if not api_key:
        load_dotenv()
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            return "⚠️ Please enter your OpenAI API key", None, None

    try:
        system = BibleLanguageLearningSystem(
            api_key=api_key,
            target_language=language,
            model=model,
        )

        lesson_content, pdf_path, audio_path = system.run_full_lesson_generation(level)
        display_text = format_lesson_display(lesson_content)
        return display_text, pdf_path, audio_path

    except Exception as e:
        return f"❌ Error: {str(e)}", None, None

In [3]:
api_key = os.getenv("OPENAI_API_KEY")
generate_lesson(api_key, "French", "B2", "gpt-4o-mini")

0.00  Starting lesson generation...
0.10  📖 Fetching verse of the day from discoverybiblestudy.org ...


[2026-02-21 05:56:14 - openai._base_client:482 - DEBUG] Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-2dd5964c-2b5a-404a-affa-c60db57c0145', 'json_data': {'messages': [{'role': 'system', 'content': 'You are a Bible study coordinator and language teacher.\nYou will receive a Bible verse in English. Your tasks:\n1. Translate the verse accurately into French.\n2. Write a short meditation paragraph (2-3 sentences) in English appropriate for B2 learners.\n3. Write the same meditation paragraph in French at B2 level (clear, educational language).\n4. Keep total word count under 1000 words.\n\nReturn ONLY valid JSON with these exact keys:\n- "verse_text_english": the English verse as provided\n- "verse_text_french": the verse translated into French\n- "meditation_english": meditation in English\n- "meditation_french": meditation in French'}, {'role': 'user', 'content': 'Verse reference: Matthew 13: 23\nEnglish verse t

0.25  📚 Creating reading comprehension...


[2026-02-21 05:56:29 - httpx:1025 - INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-02-21 05:56:29 - openai._base_client:1016 - DEBUG] HTTP Response: POST https://api.openai.com/v1/chat/completions "200 OK" Headers({'date': 'Sat, 21 Feb 2026 04:56:29 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'access-control-expose-headers': 'X-Request-ID', 'openai-organization': 'user-cdewcre4zfziz0uvslme7ssv', 'openai-processing-ms': '7086', 'openai-project': 'proj_etN2PsjYERj1c89nZSy2VicW', 'openai-version': '2020-10-01', 'server': 'cloudflare', 'x-ratelimit-limit-requests': '10000', 'x-ratelimit-limit-tokens': '200000', 'x-ratelimit-remaining-requests': '9998', 'x-ratelimit-remaining-tokens': '199811', 'x-ratelimit-reset-requests': '9.685s', 'x-ratelimit-reset-tokens': '56ms', 'x-request-id': 'req_0bcee038a9b24e6ea6b0f8d253af21ec', 'x-openai-proxy-wasm': 'v0.1', 'cf-cache-status': 'DYNAMIC', 'x-co

0.38  🎓 Designing lesson exercises...


...

[2026-02-21 05:57:14 - httpx:1025 - INFO] HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
[2026-02-21 05:57:14 - openai._base_client:1016 - DEBUG] HTTP Response: POST https://api.openai.com/v1/chat/completions "200 OK" Headers({'date': 'Sat, 21 Feb 2026 04:57:14 GMT', 'content-type': 'application/json', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'access-control-expose-headers': 'X-Request-ID', 'openai-organization': 'user-cdewcre4zfziz0uvslme7ssv', 'openai-processing-ms': '13195', 'openai-project': 'proj_etN2PsjYERj1c89nZSy2VicW', 'openai-version': '2020-10-01', 'server': 'cloudflare', 'x-ratelimit-limit-requests': '10000', 'x-ratelimit-limit-tokens': '200000', 'x-ratelimit-remaining-requests': '9995', 'x-ratelimit-remaining-tokens': '199640', 'x-ratelimit-reset-requests': '40.195s', 'x-ratelimit-reset-tokens': '108ms', 'x-request-id': 'req_cb6d1372f08c48049ffb01a30df94920', 'x-openai-proxy-wasm': 'v0.1', 'cf-cache-status': 'DYNAMIC', 'x

0.84  🔊 Generating audio...
Generating OpenAI TTS audio: lessons\reading_audio_French_B2_20260221_055714.mp3


[2026-02-21 05:57:16 - httpx:1025 - INFO] HTTP Request: POST https://api.openai.com/v1/audio/speech "HTTP/1.1 200 OK"
[2026-02-21 05:57:25 - openai._base_client:1016 - DEBUG] HTTP Response: POST https://api.openai.com/v1/audio/speech "200 OK" Headers({'date': 'Sat, 21 Feb 2026 04:57:16 GMT', 'content-type': 'audio/mpeg', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'access-control-expose-headers': 'X-Request-ID', 'openai-organization': 'user-cdewcre4zfziz0uvslme7ssv', 'openai-processing-ms': '1243', 'openai-project': 'proj_etN2PsjYERj1c89nZSy2VicW', 'openai-version': '2020-10-01', 'server': 'cloudflare', 'x-ratelimit-limit-requests': '10000', 'x-ratelimit-limit-tokens': '200000', 'x-ratelimit-remaining-requests': '9994', 'x-ratelimit-remaining-tokens': '199751', 'x-ratelimit-reset-requests': '43.779s', 'x-ratelimit-reset-tokens': '74ms', 'x-request-id': 'req_a5bc19709e0e4354a66ce284cb9424c2', 'x-openai-proxy-wasm': 'v0.1', 'cf-cache-status': 'DYNAMIC', 'x-content-type-op

0.93  📄 Creating PDF...
PDF filepath: lessons\bible_lesson_French_B2_20260221_055726.pdf
1.00  ✨ Complete!


("\n# 📖 Verse of the Day — 21st Feb 2026\n\n**Matthew 13: 23**\n\n*Les semences semées dans la bonne terre représentent des gens qui entendent le message, le comprennent et produisent une bonne récolte : certains cent, certains soixante, et certains trente fois ce qui a été semé.*\n[Read online](https://discoverybiblestudy.org/daily/d/2026-02-21?utm_source=web&medium=api&campaign=votd)\n\n**Meditation:**\nCe verset nous rappelle l'importance d'être réceptifs aux enseignements que nous recevons. Tout comme les semences ont besoin d'une bonne terre pour grandir, nous avons également besoin d'un cœur préparé pour vraiment comprendre et appliquer ces messages dans nos vies. Lorsque nous le faisons, nous pouvons créer des changements positifs et partager notre croissance avec les autres.\n\n---\n\n# 📚 Reading Comprehension\n\nDans l'Évangile selon Matthieu, le verset 13:23 évoque l'importance d'une bonne réception du message spirituel. Les semences qui tombent dans la bonne terre représente